In [ ]:
import scanpy as sc
import pandas as pd

# --- 1. Load metadata (already clean) ---

from pathlib import Path

cwd = Path.cwd()
ANALYSIS_DIR = (
    cwd
    if cwd.name == "06_b_cell_igvf"
    else Path("06_b_cell_igvf")
    if Path("06_b_cell_igvf").exists()
    else Path("..").resolve()
    if cwd.name == "notebooks"
    else Path("../..").resolve()
)
DATA_DIR = Path("data/flowmap_manuscript/b_cell")
if not DATA_DIR.exists():
    DATA_DIR = ANALYSIS_DIR.parent / "data" / "flowmap_manuscript" / "b_cell"
RESULTS_DIR = Path("data/flowmap_manuscript/b_cell_results")
if not RESULTS_DIR.exists():
    RESULTS_DIR = ANALYSIS_DIR.parent / "data" / "flowmap_manuscript" / "b_cell_results"
FIGURE_DIR = ANALYSIS_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
UTILS_DIR = ANALYSIS_DIR / "utils"

meta = pd.read_csv(DATA_DIR / "igvf12_barcode_metadata.txt", sep="\t")

# Example: AAAC..._3
meta["sequence"] = meta["barcode"].str.split("-").str[0]
meta["lane"] = meta["barcode"].str.split("-").str[1].astype(int)
meta

In [ ]:
# Group by lane
lane_to_sequences = {
    lane: set(df["sequence"])
    for lane, df in meta.groupby("lane")
}

# --- 2. Accessions ---
accessions = {
    1: "IGVFFI3928IUMP",
    2: "IGVFFI9487YVMZ",
    3: "IGVFFI3157KHGC",
    4: "IGVFFI7038ZJIA",
    5: "IGVFFI7443GEKI",
    6: "IGVFFI8484PULG",
    7: "IGVFFI4004TUTY",
}

adatas = []

# --- 3. Process each lane ---
for lane, acc in accessions.items():
    print(f"Processing lane {lane}")

    adata = sc.read_h5ad(DATA_DIR / f"{acc}.h5ad")

    # Extract DNA sequence (remove -1)
    adata.obs["sequence"] = adata.obs_names.str.split("_").str[0]

    # Filter by sequence only (since lane is implicit here)
    keep = adata.obs["sequence"].isin(lane_to_sequences.get(lane, set()))
    adata = adata[keep].copy()

    print(f"  kept {adata.n_obs} cells")

    # Optional: store lane info
    adata.obs["lane"] = lane

    adatas.append(adata)

# --- 4. Concatenate ---
adata_combined = sc.concat(adatas, join="outer")

print(adata_combined)

In [ ]:
# import numpy as np

# # Group by lane
# lane_to_sequences = {
#     lane: set(df["sequence"])
#     for lane, df in meta.groupby("lane")
# }

# # --- 2. Accessions ---
# accessions = {
#     1: "IGVFFI3928IUMP",
#     2: "IGVFFI9487YVMZ",
#     3: "IGVFFI3157KHGC",
#     4: "IGVFFI7038ZJIA",
#     5: "IGVFFI7443GEKI",
#     6: "IGVFFI8484PULG",
#     7: "IGVFFI4004TUTY",
# }

# adatas = []

# for lane, acc in accessions.items():
#     print(f"Processing lane {lane}")

#     adata = sc.read_h5ad(DATA_DIR / f"{acc}.h5ad")

#     # Metadata
#     adata.obs["sequence"] = adata.obs_names.str.split("_").str[0]
#     adata.obs["lane"] = lane

#     # QC
#     sc.pp.filter_cells(adata, min_counts=1000)
#     sc.pp.filter_cells(adata, min_genes=300)
#     sc.pp.filter_genes(adata, min_cells=10)

#     print(f"  kept {adata.n_obs} cells after QC")

#     # --- 🔽 Random 10% sampling ---
#     n_sample = int(0.05 * adata.n_obs)
#     idx = np.random.choice(adata.obs_names, size=n_sample, replace=False)
#     adata = adata[idx].copy()

#     print(f"  sampled {adata.n_obs} cells")

#     adatas.append(adata)

# # Concatenate
# adata_combined = sc.concat(adatas, join="outer")

# print(adata_combined)

In [ ]:
adata = adata_combined
# Build lookup table
meta_keyed = meta.set_index(["sequence", "lane"])

# Map
adata.obs["celltype_level1"] = [
    meta_keyed.loc[(seq, lane), "celltype_level1"]
    for seq, lane in zip(adata.obs["sequence"], adata.obs["lane"])
]

adata.obs["celltype_level2"] = [
    meta_keyed.loc[(seq, lane), "celltype_level2"]
    for seq, lane in zip(adata.obs["sequence"], adata.obs["lane"])
]

In [ ]:
# adata = adata_combined
adata.layers["spliced"] = adata.layers["mature"]
adata.layers["unspliced"] = adata.layers["nascent"]
# del adata.layers["ambiguous"]

In [ ]:
import scanpy as sc
import scvelo as scv

# Normalize
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# HVG
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat")
adata = adata[:, adata.var["highly_variable"]].copy()

# PCA
sc.pp.pca(adata, n_comps=30)

# Moments
scv.pp.moments(adata, n_neighbors=30, n_pcs=30)

# Velocity
scv.tl.velocity(adata, mode="stochastic")
scv.tl.velocity_graph(adata)

In [ ]:
# Embedding
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color="celltype_level1")
sc.pl.umap(adata, color="celltype_level2")

In [ ]:
scv.pl.velocity_embedding_stream(
    adata,
    basis="umap"
)

In [ ]:
# Time
scv.tl.velocity_pseudotime(adata)

scv.tl.recover_dynamics(adata)

scv.tl.latent_time(adata)

In [ ]:
scv.pl.umap(adata, color="velocity_pseudotime")
scv.pl.umap(adata, color="latent_time")

In [ ]:
adata.write(DATA_DIR / "bcell_velocity_processed.h5ad")

In [ ]:
adata